In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Set here the number of nodes
TOTAL_NODES = 25 

def generate_publication_ready_plots(data_file, n_nodes=TOTAL_NODES):
    if not os.path.exists(data_file):
        print(f"Error: '{data_file}' not found.")
        return
        
    print(f"Loading data from {data_file} (Targeting {n_nodes} nodes)...")
    df = pd.read_csv(data_file)

    # --- PERCENTAGE CALCULATION (Dinamico) ---
    df['pct_perturbed'] = (df['n_perturbed_nodes'] / n_nodes * 100).round(1).astype(str) + '%'
    df['pct_positives'] = (df['n_positives'] / n_nodes * 100).round(1).astype(str) + '%'

    
    # Style Configuration
    
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)
    
    COLOR_NAIVE = '#5F4690'     
    COLOR_BOOTSTRAP = '#1D6996' 
    COLOR_MAP = {'Naive': COLOR_NAIVE, 'Bootstrap': COLOR_BOOTSTRAP}

    
    # PLOT 1: Learning Curves
    
    print("Generating 'fig1_learning_curves.png'...")
    
    g1 = sns.relplot(
        data=df, 
        x='n1', y='aupr', 
        hue='method', col='edge_prob',
        kind='line', palette=COLOR_MAP, markers=True, dashes=False,
        errorbar=('ci', 95), 
        facet_kws={'sharex': False, 'sharey': True},
        height=4.5, aspect=1.2, 
        linewidth=2.5, markersize=9
    )
    
    g1.set_axis_labels("Sample Size (N)", "Mean AUPR")
    g1.set_titles(col_template="Graph Density: {col_name}", fontweight='bold')
    sns.move_legend(g1, "upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, title=None)
    
    g1.savefig("fig1_learning_curves.png", dpi=300, bbox_inches='tight')
    plt.close()

    
    # PLOT 2: Topological Stress Test
    
    print("Generating 'fig2_topological_stress.png'...")
    sparsity_levels = sorted(df['edge_prob'].unique())
    
    fig2, axes2 = plt.subplots(1, len(sparsity_levels), figsize=(6 * len(sparsity_levels), 5), sharey=True)
    if len(sparsity_levels) == 1: axes2 = [axes2]
    
    for i, sparsity in enumerate(sparsity_levels):
        sns.barplot( 
            data=df[df['edge_prob'] == sparsity].sort_values('n_perturbed_nodes'),
            x='pct_perturbed', y='aupr', hue='method', 
            palette=COLOR_MAP, ax=axes2[i], capsize=.1, errorbar=('ci', 95)
        )
        axes2[i].set_title(f"Density: {sparsity}", fontsize=14, fontweight='bold')
        axes2[i].set_xlabel("Network Perturbed (%)")
        axes2[i].set_ylabel("Mean AUPR" if i == 0 else "")
        if axes2[i].get_legend() is not None: axes2[i].get_legend().remove()
            
    handles, labels = axes2[0].get_legend_handles_labels()
    fig2.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False, fontsize=13)
    
    plt.tight_layout()
    fig2.subplots_adjust(top=0.88) 
    fig2.savefig("fig2_topological_stress.png", dpi=300, bbox_inches='tight')
    plt.close()

    
    # PLOT 3: Intervention Complexity

    print("Generating 'fig3_intervention_complexity.png'...")
    # Change .max() in case you want to select a specific density.
    df_pos = df[df['edge_prob'] == df['edge_prob'].max()] 
    
    if not df_pos.empty and len(df_pos['n_positives'].unique()) > 1:
        fig3, axes3 = plt.subplots(1, 2, figsize=(12, 5))
        
        sns.barplot(data=df_pos.sort_values('n_positives'), x='pct_positives', y='precision', hue='method', palette=COLOR_MAP, ax=axes3[0], capsize=.1)
        axes3[0].set_title("Precision vs Complexity", fontsize=14, fontweight='bold')
        
        sns.barplot(data=df_pos.sort_values('n_positives'), x='pct_positives', y='recall', hue='method', palette=COLOR_MAP, ax=axes3[1], capsize=.1)
        axes3[1].set_title("Recall vs Complexity", fontsize=14, fontweight='bold')
        
        for ax in axes3:
            ax.set_xlabel("Intervention Targets (%)")
            if ax.get_legend(): ax.get_legend().remove()
                
        handles, labels = axes3[0].get_legend_handles_labels()
        fig3.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=2, frameon=False, fontsize=13)
        
        plt.tight_layout()
        fig3.subplots_adjust(top=0.88)
        fig3.savefig("fig3_intervention_complexity.png", dpi=300, bbox_inches='tight')
        plt.close()

    print("Process complete! Publication-ready plots are saved.")

if __name__ == "__main__":
    generate_publication_ready_plots("MASTER_grid_results_25nodes.csv", n_nodes=TOTAL_NODES)

Loading data from MASTER_grid_results_25nodes.csv (Targeting 25 nodes)...
Generating 'fig1_learning_curves.png'...
Generating 'fig2_topological_stress.png'...
Generating 'fig3_intervention_complexity.png'...
Process complete! Publication-ready plots are saved.
